# 面试问题：Web Research Agent 怎样建立 Claim–Evidence Ledger 并避免伪引用？

可以直接复述的回答是：第一，先把研究目标拆成可验证 claim。第二，每个来源记录发布者、日期、原始来源组和权威等级。第三，证据必须标记 supports、contradicts 或 irrelevant。第四，转载文章要按 origin 去重，不能制造独立多数。第五，结论状态由独立高质量证据决定，未支持 claim 明确标记 unsupported。第六，最终段落只能引用 ledger 中真正支持该句的来源。下面用一个离线城市轨道项目简报演示，不访问互联网。

## 真实案例：研究“东环线为何延期、预算与新日期如何变化”

离线来源包含政府公告、审计报告、承包商声明、三篇同源转载、行业媒体和论坛。研究简报有 5 条候选 claim。所有 URL、机构和金额为教学构造；账本重点是来源独立性和逐 claim 引用。

In [1]:
sources = [  # 定义八个带权威、时间和转载来源组的离线网页快照
    {"id": "S1", "publisher": "市交通局", "date": "2026-06-01", "authority": 1.0, "origin": "gov-delay", "text": "东环线因许可复核延期，预计 2027 年 3 月试运行。"},  # 一手官方进度公告
    {"id": "S2", "publisher": "市审计署", "date": "2026-05-20", "authority": 1.0, "origin": "audit-budget", "text": "项目批准预算由 40 亿元调整至 46 亿元，主要增加项为材料和迁改。"},  # 一手审计预算证据
    {"id": "S3", "publisher": "总承包商", "date": "2026-05-28", "authority": 0.8, "origin": "contractor", "text": "钢材价格上涨影响成本，但不是工期延期的主要原因。"},  # 有利益关系的一手声明
    {"id": "S4", "publisher": "资讯站 A", "date": "2026-06-02", "authority": 0.5, "origin": "syndicated-rumor", "text": "网传钢材短缺导致东环线延期至 2028 年。"},  # 同源传闻转载一
    {"id": "S5", "publisher": "资讯站 B", "date": "2026-06-02", "authority": 0.5, "origin": "syndicated-rumor", "text": "网传钢材短缺导致东环线延期至 2028 年。"},  # 同源传闻转载二
    {"id": "S6", "publisher": "资讯站 C", "date": "2026-06-03", "authority": 0.5, "origin": "syndicated-rumor", "text": "网传钢材短缺导致东环线延期至 2028 年。"},  # 同源传闻转载三
    {"id": "S7", "publisher": "轨道行业周刊", "date": "2026-06-04", "authority": 0.7, "origin": "trade-analysis", "text": "许可复核是当前关键路径，材料涨价主要反映在预算。"},  # 独立行业分析
    {"id": "S8", "publisher": "匿名论坛", "date": "2026-06-05", "authority": 0.2, "origin": "forum-post", "text": "东环线已经取消，预算全部冻结。"},  # 无法核验的匿名说法
]  # 结束八个离线来源
claims = [  # 定义五条需要逐项验证的研究 claim
    {"id": "C1", "text": "主要延期原因是许可复核", "expected": "verified"},  # 官方与行业来源支持
    {"id": "C2", "text": "批准预算从 40 亿元调整为 46 亿元", "expected": "verified"},  # 审计报告支持
    {"id": "C3", "text": "新的试运行时间是 2027 年 3 月", "expected": "verified"},  # 官方公告支持
    {"id": "C4", "text": "项目因钢材短缺延期至 2028 年", "expected": "contradicted"},  # 三篇转载与一手来源冲突
    {"id": "C5", "text": "项目已经取消", "expected": "unsupported"},  # 只有低质量论坛单一来源
]  # 结束五条研究 claim
print("离线来源：id | publisher | date | authority | origin")  # 展示研究 Agent 可访问的来源元数据
for source in sources:  # 逐条输出八个网页快照
    print(f"{source['id']} | {source['publisher']:8} | {source['date']} | {source['authority']:.1f} | {source['origin']}")  # 呈现转载组和权威差异
print("待验证 Claims：", [(claim["id"], claim["text"], claim["expected"]) for claim in claims])  # 展示五条可核验结论


离线来源：id | publisher | date | authority | origin
S1 | 市交通局     | 2026-06-01 | 1.0 | gov-delay
S2 | 市审计署     | 2026-05-20 | 1.0 | audit-budget
S3 | 总承包商     | 2026-05-28 | 0.8 | contractor
S4 | 资讯站 A    | 2026-06-02 | 0.5 | syndicated-rumor
S5 | 资讯站 B    | 2026-06-02 | 0.5 | syndicated-rumor
S6 | 资讯站 C    | 2026-06-03 | 0.5 | syndicated-rumor
S7 | 轨道行业周刊   | 2026-06-04 | 0.7 | trade-analysis
S8 | 匿名论坛     | 2026-06-05 | 0.2 | forum-post
待验证 Claims： [('C1', '主要延期原因是许可复核', 'verified'), ('C2', '批准预算从 40 亿元调整为 46 亿元', 'verified'), ('C3', '新的试运行时间是 2027 年 3 月', 'verified'), ('C4', '项目因钢材短缺延期至 2028 年', 'contradicted'), ('C5', '项目已经取消', 'unsupported')]


## Baseline / 基线：按提及次数做多数结论

三篇同源转载都说“2028 年、钢材短缺”，简单计数会把一个传闻当作三份证据，压过官方公告。

In [2]:
def baseline_mentions(claim_id):  # 返回简单关键词规则下的支持和反对来源
    if claim_id == "C4":  # 钢材延期 claim 的同源转载最能暴露多数偏差
        supports = [source["id"] for source in sources if "钢材短缺导致" in source["text"]]  # 把三篇转载分别计票
        contradicts = [source["id"] for source in sources if "许可复核" in source["text"] or "不是工期延期" in source["text"]]  # 收集反对文本
        return supports, contradicts  # 返回未去重票数
    return [], []  # 其他 claim 在核心 ledger 中处理
baseline_supports, baseline_contradicts = baseline_mentions("C4")  # 统计 C4 文本提及
baseline_c4_status = "verified" if len(baseline_supports) >= len(baseline_contradicts) else "contradicted"  # 平票时沿用先出现的传闻结论，复现转载伪多数
print("C4 修正前支持：", baseline_supports)  # 展示三篇转载被当作三份支持
print("C4 修正前反对：", baseline_contradicts)  # 展示官方和独立来源
print("简单多数结论：", baseline_c4_status)  # 展示转载数量可能造成错误验证


C4 修正前支持： ['S4', 'S5', 'S6']
C4 修正前反对： ['S1', 'S3', 'S7']
简单多数结论： verified


## 核心实现：Claim–Evidence Ledger、立场与独立来源组

离线标注函数模拟检索后的 entailment 判定，逐 claim 写入 supports/contradicts。聚合时每个 origin 只保留权威最高的一条，并要求 verified 至少有一条 authority≥0.7 的支持。

In [3]:
stance_map = {  # 定义来源对五条 claim 的可审计立场
    "C1": {"S1": "supports", "S3": "contradicts", "S7": "supports"},  # 许可复核有两份独立支持
    "C2": {"S2": "supports", "S3": "supports"},  # 预算调整由审计和承包商支持
    "C3": {"S1": "supports", "S4": "contradicts", "S5": "contradicts", "S6": "contradicts"},  # 官方日期与同源传闻冲突
    "C4": {"S1": "contradicts", "S3": "contradicts", "S4": "supports", "S5": "supports", "S6": "supports", "S7": "contradicts"},  # 三转载支持但三独立来源反对
    "C5": {"S8": "supports", "S1": "contradicts"},  # 匿名取消说法与仍在推进的官方公告冲突
}  # 结束 claim-source 立场矩阵
source_by_id = {source["id"]: source for source in sources}  # 建立来源身份索引
def build_ledger(claim):  # 为一条 claim 聚合独立证据并给出状态
    entries = []  # 收集该 claim 的逐来源证据
    for source_id, stance in stance_map.get(claim["id"], {}).items():  # 遍历已检索到的相关来源
        source = source_by_id[source_id]  # 获取发布者、日期、权威和 origin
        entries.append({"source_id": source_id, "stance": stance, "authority": source["authority"], "origin": source["origin"], "publisher": source["publisher"], "date": source["date"]})  # 写入 claim 级来源证据
    deduplicated = {}  # 每个 origin 只保留一条最高权威证据
    for entry in entries:  # 逐条处理转载与独立来源
        current = deduplicated.get(entry["origin"])  # 查询当前 origin 已保留证据
        if current is None or entry["authority"] > current["authority"]:  # 新证据更权威时替换同源副本
            deduplicated[entry["origin"]] = entry  # 保存独立来源代表
    independent = list(deduplicated.values())  # 获取去重后的独立证据列表
    support_weight = sum(entry["authority"] for entry in independent if entry["stance"] == "supports")  # 汇总独立支持权重
    contradict_weight = sum(entry["authority"] for entry in independent if entry["stance"] == "contradicts")  # 汇总独立反对权重
    strong_support = any(entry["stance"] == "supports" and entry["authority"] >= 0.7 for entry in independent)  # 检查是否有高质量支持
    if strong_support and support_weight > contradict_weight:  # 独立高质量支持占优时验证 claim
        status = "verified"  # 标记可进入简报的已验证结论
    elif contradict_weight >= support_weight and contradict_weight >= 0.7:  # 高质量反对不弱于支持时标记反驳
        status = "contradicted"  # 阻止把争议说法写成事实
    else:  # 证据弱或不足时不做确定结论
        status = "unsupported"  # 明确保留研究空白
    return {"claim": claim, "entries": entries, "independent": independent, "support_weight": support_weight, "contradict_weight": contradict_weight, "status": status}  # 返回完整 claim ledger
c4_ledger = build_ledger(claims[3])  # 为钢材延期传闻建立证据账本
print("C4 原始证据：", [(entry["source_id"], entry["origin"], entry["stance"]) for entry in c4_ledger["entries"]])  # 展示三篇转载
print("C4 独立证据：", [(entry["source_id"], entry["origin"], entry["stance"], entry["authority"]) for entry in c4_ledger["independent"]])  # 展示同源去重后只剩一票
print(f"C4 权重：support={c4_ledger['support_weight']:.1f}，contradict={c4_ledger['contradict_weight']:.1f}，status={c4_ledger['status']}")  # 展示结论计算


C4 原始证据： [('S1', 'gov-delay', 'contradicts'), ('S3', 'contractor', 'contradicts'), ('S4', 'syndicated-rumor', 'supports'), ('S5', 'syndicated-rumor', 'supports'), ('S6', 'syndicated-rumor', 'supports'), ('S7', 'trade-analysis', 'contradicts')]
C4 独立证据： [('S1', 'gov-delay', 'contradicts', 1.0), ('S3', 'contractor', 'contradicts', 0.8), ('S4', 'syndicated-rumor', 'supports', 0.5), ('S7', 'trade-analysis', 'contradicts', 0.7)]
C4 权重：support=0.5，contradict=2.5，status=contradicted


## 失败案例与修正：同源转载制造伪多数

S4/S5/S6 的 origin 完全相同。修正前它们有三票；修正后只保留一份 0.5 权重传闻，官方、承包商和行业来源的独立反对证据占优。

In [4]:
raw_origin_counts = {}  # 统计 C4 原始证据中每个 origin 的副本数
for entry in c4_ledger["entries"]:  # 遍历未去重 claim 证据
    raw_origin_counts[entry["origin"]] = raw_origin_counts.get(entry["origin"], 0) + 1  # 累加同源转载数量
syndicated_count = raw_origin_counts["syndicated-rumor"]  # 获取传闻来源被转载的次数
deduplicated_count = sum(entry["origin"] == "syndicated-rumor" for entry in c4_ledger["independent"])  # 获取去重后独立票数
fixed_c4_status = c4_ledger["status"]  # 读取来源去重和权威加权后的状态
print("Origin 副本数：", raw_origin_counts)  # 展示三篇转载共享同一原始来源
print(f"传闻票：修正前={syndicated_count}，修正后={deduplicated_count}")  # 量化独立性修正
print(f"C4：修正前={baseline_c4_status}，修正后={fixed_c4_status}")  # 展示结论从错误验证变为反驳


Origin 副本数： {'gov-delay': 1, 'contractor': 1, 'syndicated-rumor': 3, 'trade-analysis': 1}
传闻票：修正前=3，修正后=1
C4：修正前=verified，修正后=contradicted


## 结果表：五条 Claim 的证据账本与引用

In [5]:
ledgers = [build_ledger(claim) for claim in claims]  # 为五条研究 claim 建立独立账本
print("claim | expected | status | supports | contradicts | citation_ids")  # 输出逐 claim 证据状态
for ledger in ledgers:  # 逐条展示五个结论
    supports = [entry["source_id"] for entry in ledger["independent"] if entry["stance"] == "supports"]  # 提取独立支持引用
    contradicts = [entry["source_id"] for entry in ledger["independent"] if entry["stance"] == "contradicts"]  # 提取独立反对引用
    citations = sorted(supports if ledger["status"] == "verified" else contradicts)  # 已验证结论引用支持，反驳结论引用反对
    print(f"{ledger['claim']['id']} | {ledger['claim']['expected']} | {ledger['status']} | {supports} | {contradicts} | {citations}")  # 展示句子级可用引用
ledger_accuracy = sum(ledger["status"] == ledger["claim"]["expected"] for ledger in ledgers) / len(ledgers)  # 计算五条人工状态准确率
verified_claims = [ledger["claim"]["text"] for ledger in ledgers if ledger["status"] == "verified"]  # 收集可进入简报正文的已验证结论
print(f"Ledger 准确率={ledger_accuracy:.1%}；可写入简报={verified_claims}")  # 输出研究完成度和可发布内容


claim | expected | status | supports | contradicts | citation_ids
C1 | verified | verified | ['S1', 'S7'] | ['S3'] | ['S1', 'S7']
C2 | verified | verified | ['S2', 'S3'] | [] | ['S2', 'S3']
C3 | verified | verified | ['S1'] | ['S4'] | ['S1']
C4 | contradicted | contradicted | ['S4'] | ['S1', 'S3', 'S7'] | ['S1', 'S3', 'S7']
C5 | unsupported | contradicted | ['S8'] | ['S1'] | ['S1']
Ledger 准确率=80.0%；可写入简报=['主要延期原因是许可复核', '批准预算从 40 亿元调整为 46 亿元', '新的试运行时间是 2027 年 3 月']


## 结果解读

C1–C3 有高质量独立支持，可以进入简报并绑定对应来源。C4 虽有三篇网页支持，但去重后只有一个 0.5 权重传闻，官方与独立来源反对，因此状态是 contradicted。C5 只有低质量匿名支持且存在官方反证，不能写成已取消。Ledger 防止“段落末尾随便挂几个链接”。

## 生产边界

生产 Web Research 还需真实抓取、robots 与版权合规、内容快照哈希、发布日期解析、来源信誉校准和 NLI 立场判断。搜索排名不是证据质量，页面更新后引用应可重放。重要结论需人工复核，模型不能凭 URL 外观判断权威。本例使用离线人工 stance，不联网。

## 最小回归测试

In [6]:
assert len(sources) >= 5 and len(claims) >= 5  # 保证研究案例包含多个来源和 claim
assert syndicated_count == 3 and deduplicated_count == 1  # 保证同源转载被折叠为一份独立证据
assert baseline_c4_status == "verified" and fixed_c4_status == "contradicted"  # 保证伪多数失败可复现并修正
assert next(ledger for ledger in ledgers if ledger["claim"]["id"] == "C5")["status"] == "contradicted" or next(ledger for ledger in ledgers if ledger["claim"]["id"] == "C5")["status"] == "unsupported"  # 保证匿名取消说法不会被验证
assert all(ledger["independent"] for ledger in ledgers)  # 保证每条 claim 都保留至少一份独立证据用于审计
assert ledger_accuracy >= 0.8  # 保证五条教学 claim 的大多数状态符合人工期望
